# Phase 1 — Data Generation for Pivot Research (Kaggle)

**Goal:** Generate model outputs on prompts likely to trigger reasoning pivots ("Actually," "Wait," "Hmm let me reconsider"). Save raw outputs locally to `/kaggle/working/` for Phase 2 (pivot detection + labeling).

**Runtime:** Qwen2.5-1.5B-Instruct on T4 or P100 GPU. ~30 prompts × 15 samples = 450 generations. Expect ~30–60 min on T4/P100, 

**Before running:**
1. Session options (top right) → Accelerator → GPU T4 x2 (or P100)
2. Run cells in order
3. After completion, your raw outputs will be at `/kaggle/working/pivot_research/raw_outputs.jsonl`
4. Download via: Kaggle sidebar → Output → Download `raw_outputs.jsonl`

## 1. Verify GPU and load model

We now use **Qwen2.5-1.5B-Instruct** instead of Pythia-1.4B because:
- Pythia is a *base* model — it doesn't follow instructions or reason step-by-step
- Qwen2.5-1.5B-Instruct is instruction-tuned, so it actually engages with the problem
- Same size (fits on T4 with room to spare), but produces real chain-of-thought reasoning
- No Hugging Face login required — fully open weights
- TransformerLens support is available for Phase 3

If you have a T4 x2 or P100, you can upgrade to `Qwen/Qwen2.5-7B-Instruct` for even higher pivot rates (~35–50%).

In [6]:
import torch, json, os, re, time
from datetime import datetime
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm

assert torch.cuda.is_available(), 'No GPU — Session options → Accelerator → GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map='auto'
)
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'
print('Model loaded')

GPU: Tesla T4


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded


## 2. The prompt set

30 prompts across three categories. Each prompt now uses a **two-part scaffold**:

1. A system message that gives the model *explicit permission* to change its mind
2. A user message with the problem + `"Think step by step. Double-check your reasoning before concluding."`

This two-part approach is critical: instruction-tuned models are trained to be confident.
Without an explicit nudge to self-doubt, they rarely pivot even when their first answer is wrong.
The system prompt lowers that threshold.

In [7]:
# This is the exact config that produced 9.2% pivot rate / 14 corrections.
# Deliberately NOT including any few-shot text.
SYSTEM_PROMPT = """You are a careful, honest reasoner. When solving a problem:
1. Start reasoning immediately — commit to a first approach even if uncertain
2. Work through the steps out loud
3. Pause mid-way and ask: 'Wait, is this actually right?'
4. If you catch ANY error or realize something was off, say so explicitly using phrases like:
   'Wait,' 'Actually,' 'Hmm, let me reconsider,' 'I made an error,' 'Oh wait'
5. Correct yourself and give the final answer

Never skip steps. Never suppress doubt. Thinking out loud is the goal."""

def make_prompt(question: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question + "\n\nRemember: commit to a first answer, then double-check it. Show your doubt explicitly."},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Verify no bug
print(make_prompt("test")[:400])
print("\n--- Should NOT contain the literal text '{FEW_SHOT_EXAMPLE}' above ---")

<|im_start|>system
You are a careful, honest reasoner. When solving a problem:
1. Start reasoning immediately — commit to a first approach even if uncertain
2. Work through the steps out loud
3. Pause mid-way and ask: 'Wait, is this actually right?'
4. If you catch ANY error or realize something was off, say so explicitly using phrases like:
   'Wait,' 'Actually,' 'Hmm, let me reconsider,' 'I made

--- Should NOT contain the literal text '{FEW_SHOT_EXAMPLE}' above ---


In [8]:
PROMPTS = [
    {"id": "mis2_01", "category": "misdirection", "text": "A shepherd has 25 goats. All but 8 wander off. How many goats does he have left?"},
    {"id": "mis2_02", "category": "misdirection", "text": "You have 15 candies. You give away all but 5. How many do you have?"},
    {"id": "mis2_03", "category": "misdirection", "text": "A baker makes 40 loaves. All but 12 are sold. How many loaves are unsold?"},
    {"id": "mis2_04", "category": "misdirection", "text": "There are 8 fish in a tank. 3 of them drown. How many fish are left?"},
    {"id": "mis2_05", "category": "misdirection", "text": "A librarian has 60 books. All but 20 are checked out. How many are still on the shelf?"},
    {"id": "mis2_06", "category": "misdirection", "text": "You find a diamond in the desert and put it in your pocket, which has a hole in it. You keep walking for an hour. Do you still have the diamond?"},
    {"id": "mis2_07", "category": "misdirection", "text": "A man builds a house with four walls, each one facing south. A bear walks past the house. What color is the bear?"},
    {"id": "mis2_08", "category": "misdirection", "text": "A plane crashes exactly on the equator. In which hemisphere do they bury the survivors?"},
    {"id": "mis2_09", "category": "misdirection", "text": "You have 30 marbles. You lose all but 9. How many marbles remain?"},
    {"id": "mis2_10", "category": "misdirection", "text": "A grocer has 50 apples. All but 14 are rotten. How many good apples does he have?"},
    {"id": "mis2_11", "category": "misdirection", "text": "There are 6 eggs in a basket. You take away 4. How many eggs do you have?"},
    {"id": "mis2_12", "category": "misdirection", "text": "A man pushes his car to a hotel and, upon arriving, instantly loses everything he owns. What happened?"},
    {"id": "mis2_13", "category": "misdirection", "text": "How many months of the year have exactly 30 days?"},
    {"id": "mis2_14", "category": "misdirection", "text": "Before Mount Everest was discovered, what was the tallest mountain in the world?"},
    {"id": "mis2_15", "category": "misdirection", "text": "A truck driver is going the wrong way down a one-way street. A police officer sees him clearly but does not stop him. Why?"},
    {"id": "mis2_16", "category": "misdirection", "text": "A farmer had 12 cows. All but 5 died of a disease. How many cows does he have left?"},
    {"id": "mis2_17", "category": "misdirection", "text": "You have 3 apples and you take away 2. How many apples do you have?"},
    {"id": "mis2_18", "category": "misdirection", "text": "A clerk had 45 pencils. All but 13 broke. How many unbroken pencils remain?"},
]

print(f'New prompts: {len(PROMPTS)}')

New prompts: 18


## 5. Sampling configuration

- `SAMPLES_PER_PROMPT = 15` — middle of the 10–20 range from the plan
- `temperature=0.8` — high enough for diverse reasoning, low enough to stay coherent
- `max_new_tokens=300` — most pivots happen in the first ~200 tokens; 300 gives headroom
- `top_p=0.95` — standard nucleus sampling

Tweak `SAMPLES_PER_PROMPT` down to 10 if you want to iterate faster, up to 20 for the final run.

In [9]:
SAMPLES_PER_PROMPT = 15
TEMPERATURE = 0.9
TOP_P = 0.95
MAX_NEW_TOKENS = 400
BATCH_SIZE = 8

OUTPUT_PATH = '/kaggle/working/pivot_research/raw_outputs_batch2.jsonl'
os.makedirs('/kaggle/working/pivot_research', exist_ok=True)

# This is a NEW file — does not touch your existing raw_outputs.jsonl
if os.path.exists(OUTPUT_PATH):
    os.remove(OUTPUT_PATH)
    print('Cleared old batch2 file')

TOTAL = len(PROMPTS) * SAMPLES_PER_PROMPT
print(f'Will generate {TOTAL} outputs ({len(PROMPTS)} prompts x {SAMPLES_PER_PROMPT} samples)')

Will generate 270 outputs (18 prompts x 15 samples)


## 6. Generation loop

Saves each generation as a JSONL line immediately after it's produced. This means **if Colab disconnects mid-run, you don't lose what's already done** — you can resume from where you left off (the loop checks for already-completed prompt_ids).

Each saved record contains:
- `prompt_id`, `category`, `prompt_text`
- `sample_idx` (0 to SAMPLES_PER_PROMPT-1)
- `generated_text` — only the new tokens, not the prompt
- `full_text` — prompt + generation, for context
- `model_name`, `temperature`, `top_p`, `seed`
- `timestamp`

In [10]:
def generate_batch(items):
    torch.manual_seed(items[0]['seed'])
    prompts = [make_prompt(item['prompt_text']) for item in items]
    inputs = tokenizer(prompts, return_tensors='pt', padding=True, truncation=True, max_length=512).to('cuda')
    input_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE,
            top_p=TOP_P, do_sample=True, pad_token_id=tokenizer.pad_token_id,
        )
    return [tokenizer.decode(out[input_len:], skip_special_tokens=True) for out in outputs]

all_jobs = [
    {'prompt_id': p['id'], 'category': p['category'], 'prompt_text': p['text'],
     'sample_idx': s, 'seed': hash((p['id'], s)) % (2**31)}
    for p in PROMPTS for s in range(SAMPLES_PER_PROMPT)
]

start = time.time()
with open(OUTPUT_PATH, 'a') as fout:
    pbar = tqdm(total=len(all_jobs), desc='Generating batch 2')
    for i in range(0, len(all_jobs), BATCH_SIZE):
        batch = all_jobs[i:i+BATCH_SIZE]
        try:
            gens = generate_batch(batch)
        except RuntimeError as e:
            if 'CUDA out of memory' in str(e):
                print(f'OOM — reduce BATCH_SIZE and rerun')
                break
            raise
        for job, gen in zip(batch, gens):
            fout.write(json.dumps({
                'prompt_id': job['prompt_id'], 'category': job['category'],
                'prompt_text': job['prompt_text'], 'sample_idx': job['sample_idx'],
                'generated_text': gen, 'full_text': job['prompt_text'] + '\n\n' + gen,
                'model_name': MODEL_NAME, 'temperature': TEMPERATURE, 'top_p': TOP_P,
                'seed': job['seed'], 'timestamp': datetime.now().isoformat(),
            }) + '\n')
        fout.flush()
        os.fsync(fout.fileno())
        pbar.update(len(batch))
    pbar.close()

print(f'\nDone in {(time.time()-start)/60:.1f} min')

Generating batch 2:   0%|          | 0/270 [00:00<?, ?it/s]


Done in 13.3 min


## 7. Quick sanity check

Look at a few samples and count how many already contain pivot phrases. This gives you an early read on whether the prompts are working — if you see <5% pivot rate, you'll want to either swap to a stronger model or revise the prompts before doing Phase 2.

In [1]:
import re, json
from collections import Counter

OUTPUT_PATH = '/kaggle/working/pivot_research/raw_outputs_batch2.jsonl'

PIVOT_PATTERNS = [
    r"\bwait,\s", r"\bwait—",
    r"\bactually,\s+(?:no|let|i|that|this|the|we|my|on)",
    r"\bhmm,\s", r"\bi made an error", r"\bi was wrong",
    r"\bthat('s| is) (wrong|incorrect|not right)",
    r"\blet me reconsider", r"\blet me recalculate",
    r"\blet me re-?read", r"\bon second thought",
    r"\bi need to correct", r"\bno,\s+wait", r"\bhold on",
    r"\bi('ve| have) made a mistake",
    r"\bactually,\s+i\b", r"\bactually,\s+no\b", r"\bactually,\s+that\b",
    # meta-cognitive
    r"\bmy reasoning approach was flawed",
    r"\bi misunderstood the problem",
    r"\bi need to rethink my entire approach",
    r"\bmy whole framework",
    r"\bi was thinking about this the wrong way",
    r"\bthis isn't a math problem",
    r"\bi approached this incorrectly",
    r"\bthe question is actually asking",
]
PIVOT_RE = re.compile("|".join(PIVOT_PATTERNS), re.IGNORECASE)

with open(OUTPUT_PATH, 'r') as f:
    records = [json.loads(line) for line in f]

print('Pivot rate by category:')
for cat in ['misdirection', 'math', 'logic', 'meta_cognitive']:
    cat_records = [r for r in records if r['category'] == cat]
    pivots      = [r for r in cat_records if PIVOT_RE.search(r['generated_text'])]
    print(f'  {cat:<20} {len(pivots):>3}/{len(cat_records)}  ({len(pivots)/len(cat_records):.1%})')

print()
print('Sample meta_cognitive output:')
meta = [r for r in records if r['category'] == 'meta_cognitive']
if meta:
    print(meta[0]['generated_text'][:400])
    

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/pivot_research/raw_outputs.jsonl'

## 6. Download your output

Your file is saved at `/kaggle/working/pivot_research/raw_outputs_batch2.jsonl`.

**To download:**
- Kaggle sidebar (right side) → **Output** tab → find `raw_outputs.jsonl` → click **Download**

Alternatively, run the cell below to print the file path and size for confirmation.

In [11]:
import re, json

OUTPUT_PATH = '/kaggle/working/pivot_research/raw_outputs_batch2.jsonl'

PIVOT_PATTERNS = [
    r"\bwait,\s", r"\bwait—", r"\bactually,\s+(?:no|let|i|that|this|the|we|my|on)",
    r"\bhmm,\s", r"\bi made an error", r"\bi was wrong",
    r"\bthat('s| is) (wrong|incorrect|not right)", r"\blet me reconsider",
    r"\blet me recalculate", r"\blet me re-?read", r"\bon second thought",
    r"\bi need to correct", r"\bno,\s+wait", r"\bhold on",
    r"\bi('ve| have) made a mistake", r"\bactually,\s+i\b", r"\bactually,\s+no\b", r"\bactually,\s+that\b",
]
PIVOT_RE = re.compile("|".join(PIVOT_PATTERNS), re.IGNORECASE)
records = [json.loads(l) for l in open(OUTPUT_PATH)]
pivots = [r for r in records if PIVOT_RE.search(r['generated_text'])]
print(f'Total: {len(records)}  |  Pivots: {len(pivots)}  ({len(pivots)/len(records):.1%})')

FILTERED_PATH = '/kaggle/working/pivot_research/pivot_candidates_batch2.jsonl'
with open(FILTERED_PATH, 'w') as f:
    for r in pivots:
        f.write(json.dumps(r) + '\n')
print(f'Saved to {FILTERED_PATH}')

Total: 270  |  Pivots: 62  (23.0%)
Saved to /kaggle/working/pivot_research/pivot_candidates_batch2.jsonl
